In [0]:

from pyspark.sql import functions as F

# ADLS Gen2 Bronze source
bronze_path = (
    "abfss://bronze@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

# Read the existing CDC JSONL batch
df_bronze = (
    spark.read
    .option("multiLine", "false")
    .json(bronze_path)
)

print("Bronze columns:")
print(df_bronze.columns)

print("Bronze record count:", df_bronze.count())

display(df_bronze.limit(5))


In [0]:


from pyspark.sql import functions as F

df_bronze.select(
    "topic",
    "partition",
    "offset",
    "is_tombstone",
    F.col("event.payload.op").alias("operation"),
    F.col("event.payload.ts_ms").alias("event_timestamp"),
    F.col("event.payload.before").alias("before"),
    F.col("event.payload.after").alias("after")
).orderBy("offset").show(20, truncate=False)


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Extract Debezium payload and customer row images
df_events = df_bronze.select(
    "topic",
    "partition",
    "offset",
    "is_tombstone",
    F.col("event.payload.op").alias("op"),
    F.col("event.payload.ts_ms").alias("event_ts_ms"),
    F.col("event.payload.after.customer_id").alias("after_id"),
    F.col("event.payload.after.customer_name").alias("customer_name"),
    F.col("event.payload.after.city").alias("city"),
    F.col("event.payload.after.updated_at").alias("updated_at"),
    F.col("event.payload.before.customer_id").alias("before_id")
)

# Use the after image for inserts/updates/snapshots,
# and the before image to identify deleted customers.
df_events = (
    df_events
    .withColumn(
        "customer_id",
        F.coalesce("after_id", "before_id")
    )
    .filter(~F.col("is_tombstone"))
)

# Show normalized events in Kafka offset order
display(
    df_events.select(
        "offset", "customer_id", "op",
        "customer_name", "city", "updated_at"
    ).orderBy("offset")
)


In [0]:

# Order events by Kafka offset within each partition and topic
event_window = (
    Window
    .partitionBy("topic", "partition", "customer_id")
    .orderBy(F.col("offset").desc())
)

# Keep the newest non-tombstone event for each customer
df_latest = (
    df_events
    .withColumn("rn", F.row_number().over(event_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(
    df_latest.select(
        "customer_id",
        "customer_name",
        "city",
        "op",
        "offset"
    ).orderBy("customer_id")
)


In [0]:

# Inspect the actual field names in the delete event
display(
    df_bronze
    .filter(F.col("event.payload.op") == "d")
    .select(
        "offset",
        F.to_json(F.col("event.payload.before")).alias("before_json")
    )
)


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Rebuild normalized events with the delete key correctly extracted
df_events = df_bronze.select(
    "topic",
    "partition",
    "offset",
    "is_tombstone",
    F.col("event.payload.op").alias("op"),
    F.col("event.payload.ts_ms").alias("event_ts_ms"),
    F.col("event.payload.after.customer_id").alias("after_id"),
    F.col("event.payload.after.customer_name").alias("customer_name"),
    F.col("event.payload.after.city").alias("city"),
    F.col("event.payload.after.updated_at").alias("updated_at"),
    F.get_json_object(
        F.to_json(F.col("event.payload.before")),
        "$.customer_id"
    ).cast("long").alias("before_id")
)

df_events = (
    df_events
    .withColumn("customer_id", F.coalesce("after_id", "before_id"))
    .filter(~F.col("is_tombstone"))
)

# Keep the newest event for each customer
event_window = (
    Window
    .partitionBy("topic", "partition", "customer_id")
    .orderBy(F.col("offset").desc())
)

df_latest = (
    df_events
    .filter(F.col("customer_id").isNotNull())
    .withColumn("rn", F.row_number().over(event_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(
    df_latest.select(
        "customer_id", "customer_name", "city", "op", "offset"
    ).orderBy("customer_id")
)


In [0]:

# Inspect the inferred schema for the delete event
df_bronze.filter(
    F.col("event.payload.op") == "d"
).select(
    "event.payload.before"
).printSchema()

# Compare direct field extraction with JSON extraction
display(
    df_bronze.filter(
        F.col("event.payload.op") == "d"
    ).select(
        F.col("event.payload.before.customer_id").alias("direct_id"),
        F.get_json_object(
            F.to_json(F.col("event.payload.before")),
            "$.customer_id"
        ).alias("json_id"),
        F.to_json(F.col("event.payload.before")).alias("before_json")
    )
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Extract CDC events and recover the key for delete events
df_events = df_bronze.select(
    "topic", "partition", "offset", "is_tombstone",
    F.col("event.payload.op").alias("op"),
    F.coalesce(
        F.col("event.payload.after.customer_id"),
        F.get_json_object(
            F.to_json(F.col("event.payload.before")), "$.customer_id"
        ).cast("long")
    ).alias("customer_id"),
    F.col("event.payload.after.customer_name").alias("customer_name"),
    F.col("event.payload.after.city").alias("city"),
    F.col("event.payload.after.updated_at").alias("updated_at")
).filter(~F.coalesce(F.col("is_tombstone"), F.lit(False)))

# Keep the latest event for each customer
window = Window.partitionBy("topic", "partition", "customer_id").orderBy(
    F.col("offset").desc()
)

df_silver = (
    df_events
    .filter(F.col("customer_id").isNotNull())
    .withColumn("rn", F.row_number().over(window))
    .filter((F.col("rn") == 1) & (F.col("op") != "d"))
    .select("customer_id", "customer_name", "city", "updated_at")
    .orderBy("customer_id")
)

display(df_silver)

In [0]:
silver_table = "default.silver_customers"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

print(f"Silver Delta table created: {silver_table}")
display(spark.table(silver_table))

In [0]:
from pyspark.sql import functions as F

df_check = spark.table("default.silver_customers")

df_check.select(
    F.count("*").alias("total_customers"),
    F.countDistinct("customer_id").alias("unique_customer_ids")
).show()

df_check.orderBy("customer_id").show(truncate=False)

In [0]:
silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

(
    spark.table("default.silver_customers")
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

print("Silver Delta data saved successfully to ADLS Gen2.")

In [0]:
silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

df_silver_check = spark.read.format("delta").load(silver_path)

print(f"Silver record count: {df_silver_check.count()}")
display(df_silver_check.orderBy("customer_id"))

## Read only the incremental file

In [0]:
from pyspark.sql import functions as F

incremental_path = (
    "abfss://bronze@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/incremental/customers_2026-10-20.jsonl"
)

df_incremental = (
    spark.read
    .option("multiLine", "false")
    .json(incremental_path)
)

display(
    df_incremental.select(
        "topic", "partition", "offset", "key",
        "event.payload.op",
        "event.payload.after"
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

incremental_path = (
    "abfss://bronze@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/incremental/"
)

df_cdc = spark.read.json(incremental_path)

# Parse event and key JSON strings into structs
event_schema = """
struct<
  payload:struct<
    op:string,
    before:string,
    after:string
  >
>
"""

key_schema = """
struct<
  payload:struct<customer_id:int>
>
"""

df_parsed = (
    df_cdc
    .withColumn("parsed_event", F.from_json(F.to_json("event"), event_schema))
    .withColumn("parsed_key", F.from_json(F.to_json("key"), key_schema))
)

# Parse before/after JSON strings separately
customer_schema = """
struct<
  customer_id:int,
  customer_name:string,
  city:string,
  updated_at:long
>
"""

df_events = (
    df_parsed
    .withColumn("before_data", F.from_json("parsed_event.payload.before", customer_schema))
    .withColumn("after_data", F.from_json("parsed_event.payload.after", customer_schema))
    .select(
        F.col("parsed_event.payload.op").alias("op"),
        F.coalesce(
            F.col("after_data.customer_id"),
            F.col("before_data.customer_id"),
            F.col("parsed_key.payload.customer_id")
        ).alias("customer_id"),
        F.col("after_data.customer_name").alias("customer_name"),
        F.col("after_data.city").alias("city"),
        F.col("after_data.updated_at").alias("updated_at"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset")
    )
)

# Keep the newest event per customer in this incremental batch
w = Window.partitionBy("customer_id").orderBy(
    F.col("kafka_partition").desc(),
    F.col("kafka_offset").desc()
)

df_latest = (
    df_events
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(df_latest)

In [0]:
silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

silver = DeltaTable.forPath(spark, silver_path)

# This batch contains the latest event per customer.
# Deduplicate by Kafka partition and offset before merging.
w = Window.partitionBy("customer_id").orderBy(
    F.col("kafka_partition").desc(),
    F.col("kafka_offset").desc()
)

df_latest = (
    df_events
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Apply delete events.
df_deletes = (
    df_latest
    .filter(F.col("op") == "d")
    .select("customer_id")
)

if df_deletes.limit(1).count() > 0:
    (
        silver.alias("t")
        .merge(df_deletes.alias("s"), "t.customer_id = s.customer_id")
        .whenMatchedDelete()
        .execute()
    )

# Apply create, update and read/snapshot events.
df_upserts = (
    df_latest
    .filter(F.col("op").isin("c", "u", "r"))
    .select("customer_id", "customer_name", "city", "updated_at")
)

(
    silver.alias("t")
    .merge(df_upserts.alias("s"), "t.customer_id = s.customer_id")
    .whenMatchedUpdate(set={
        "customer_name": "s.customer_name",
        "city": "s.city",
        "updated_at": "s.updated_at"
    })
    .whenNotMatchedInsert(values={
        "customer_id": "s.customer_id",
        "customer_name": "s.customer_name",
        "city": "s.city",
        "updated_at": "s.updated_at"
    })
    .execute()
)

df_silver = spark.read.format("delta").load(silver_path)

display(df_silver.orderBy("customer_id"))
print("Total Silver rows:", df_silver.count())

In [0]:
df_silver = spark.read.format("delta").load(silver_path)

display(df_silver.orderBy("customer_id"))

print("Total rows:", df_silver.count())
print(
    "Distinct customer IDs:",
    df_silver.select("customer_id").distinct().count()
)

In [0]:
from pyspark.sql import functions as F

incremental_path = (
    "abfss://bronze@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/incremental/customers_update_2026-10-20.jsonl"
)

df_new = spark.read.json(incremental_path)

display(
    df_new.select(
        "partition",
        "offset",
        "event.payload.op",
        "event.payload.before",
        "event.payload.after"
    )
)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

customer_schema = """
struct<
    customer_id:int,
    customer_name:string,
    city:string,
    updated_at:long
>
"""

df_updates = (
    df_new
    .withColumn(
        "after_data",
        F.col("event.payload.after")
    )
    .select(
        F.col("event.payload.op").alias("op"),
        F.col("after_data.customer_id").alias("customer_id"),
        F.col("after_data.customer_name").alias("customer_name"),
        F.col("after_data.city").alias("city"),
        F.col("after_data.updated_at").alias("updated_at"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset")
    )
    .filter(
        F.col("op").isin("c", "u", "r") &
        F.col("customer_id").isNotNull()
    )
)

silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

silver = DeltaTable.forPath(spark, silver_path)

(
    silver.alias("t")
    .merge(
        df_updates.alias("s"),
        "t.customer_id = s.customer_id"
    )
    .whenMatchedUpdate(set={
        "customer_name": "s.customer_name",
        "city": "s.city",
        "updated_at": "s.updated_at"
    })
    .whenNotMatchedInsert(values={
        "customer_id": "s.customer_id",
        "customer_name": "s.customer_name",
        "city": "s.city",
        "updated_at": "s.updated_at"
    })
    .execute()
)

display(
    spark.read.format("delta")
    .load(silver_path)
    .filter(F.col("customer_id") == 6)
)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

ledger_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/metadata/processed_events/"
)

ledger_schema = """
topic STRING,
kafka_partition INT,
kafka_offset BIGINT,
customer_id INT,
operation STRING,
processed_at TIMESTAMP
"""

if not DeltaTable.isDeltaTable(spark, ledger_path):
    (
        spark.createDataFrame([], ledger_schema)
        .write
        .format("delta")
        .mode("overwrite")
        .save(ledger_path)
    )

print("CDC event ledger is ready.")

In [0]:
df_ledger = spark.read.format("delta").load(ledger_path)

display(df_ledger)

In [0]:
df_ledger = spark.read.format("delta").load(ledger_path)

df_ledger.printSchema()
print("Ledger row count:", df_ledger.count())

In [0]:
from pyspark.sql import functions as F

# Read the new incremental Bronze event
incremental_path = (
    "abfss://bronze@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/incremental/customers_update_2026-10-20.jsonl"
)

df_new = spark.read.json(incremental_path)

# Extract event metadata
df_events = df_new.select(
    F.col("topic").alias("topic"),
    F.col("partition").cast("int").alias("kafka_partition"),
    F.col("offset").cast("long").alias("kafka_offset"),
    F.col("key.payload.customer_id").cast("int").alias("customer_id"),
    F.col("event.payload.op").alias("operation"),
    F.current_timestamp().alias("processed_at")
)

# Check if this exact Kafka event is already recorded
df_unprocessed = df_events.alias("s").join(
    spark.read.format("delta").load(ledger_path).alias("t"),
    (
        (F.col("s.topic") == F.col("t.topic")) &
        (F.col("s.kafka_partition") == F.col("t.kafka_partition")) &
        (F.col("s.kafka_offset") == F.col("t.kafka_offset"))
    ),
    "left_anti"
)

display(df_unprocessed)

In [0]:

df_new.select(
    "partition",
    "offset",
    "event.payload.op",
    "event.payload.after.customer_id",
    "event.payload.after.customer_name",
    "event.payload.after.city"
).show(truncate=False)


In [0]:

silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

df_silver = spark.read.format("delta").load(silver_path)

df_silver.printSchema()


In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

# Add tracking columns without deleting existing customer data
spark.sql(f"""
    ALTER TABLE delta.`{silver_path}`
    ADD COLUMNS (
        kafka_partition INT,
        kafka_offset BIGINT
    )
""")

print("Silver schema updated with Kafka tracking columns.")


In [0]:

spark.read.format("delta").load(silver_path).printSchema()


In [0]:

from pyspark.sql import functions as F

df_silver = spark.read.format("delta").load(silver_path)

display(
    df_silver
    .select(
        "customer_id",
        "customer_name",
        "city",
        "kafka_partition",
        "kafka_offset"
    )
    .orderBy("customer_id")
)


In [0]:

print("Source event schema:")
df_event.printSchema()

print("Silver target schema:")
spark.read.format("delta").load(silver_path).printSchema()


In [0]:

from pyspark.sql import functions as F

df_event_aligned = df_event.select(
    F.col("topic").cast("string").alias("topic"),
    F.col("kafka_partition").cast("int").alias("kafka_partition"),
    F.col("kafka_offset").cast("long").alias("kafka_offset"),
    F.col("operation").cast("string").alias("operation"),
    F.col("customer_id").cast("long").alias("customer_id"),
    F.col("customer_name").cast("string").alias("customer_name"),
    F.col("city").cast("string").alias("city"),
    F.col("updated_at").cast("long").alias("updated_at")
)

df_event_aligned.printSchema()
display(df_event_aligned)


In [0]:

from pyspark.sql import functions as F

ledger_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/metadata/processed_events/"
)

df_ledger = spark.read.format("delta").load(ledger_path)

display(
    df_ledger.filter(
        (F.col("topic") == "retail.public.customers") &
        (F.col("kafka_partition") == 0) &
        (F.col("kafka_offset") == 11)
    )
)


In [0]:

from pyspark.sql import functions as F

silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

df_silver = spark.read.format("delta").load(silver_path)

display(
    df_silver
    .filter(F.col("customer_id") == 6)
    .select(
        "customer_id",
        "customer_name",
        "city",
        "kafka_partition",
        "kafka_offset"
    )
)


In [0]:

from pyspark.sql import functions as F

ledger_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/metadata/processed_events/"
)

# Confirm the Silver result before repairing the ledger
silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

silver_check = (
    spark.read.format("delta").load(silver_path)
    .filter(
        (F.col("customer_id") == 6) &
        (F.col("kafka_partition") == 0) &
        (F.col("kafka_offset") == 11) &
        (F.col("city") == "Mysuru")
    )
)

if silver_check.count() != 1:
    raise ValueError("Silver verification failed; ledger was not changed.")

ledger = spark.read.format("delta").load(ledger_path)

already_recorded = ledger.filter(
    (F.col("topic") == "retail.public.customers") &
    (F.col("kafka_partition") == 0) &
    (F.col("kafka_offset") == 11)
).limit(1).count()

if not already_recorded:
    repair_event = spark.createDataFrame(
        [(
            "retail.public.customers",
            0,
            11,
            6,
            "u"
        )],
        """
        topic STRING,
        kafka_partition INT,
        kafka_offset BIGINT,
        customer_id INT,
        operation STRING
        """
    ).withColumn("processed_at", F.current_timestamp())

    (
        repair_event.write
        .format("delta")
        .mode("append")
        .save(ledger_path)
    )

print("Ledger repair completed.")


In [0]:

from pyspark.sql import functions as F

ledger_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/metadata/processed_events/"
)

silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

# Check the event identity in the ledger
df_ledger = spark.read.format("delta").load(ledger_path)

event_count = df_ledger.filter(
    (F.col("topic") == "retail.public.customers") &
    (F.col("kafka_partition") == 0) &
    (F.col("kafka_offset") == 11)
).count()

# Check the corresponding Silver record
df_customer = (
    spark.read.format("delta").load(silver_path)
    .filter(F.col("customer_id") == 6)
    .select("customer_id", "customer_name", "city",
            "kafka_partition", "kafka_offset")
)

print("Ledger entries for offset 11:", event_count)
display(df_customer)

if event_count == 1:
    print("PASS: exactly one ledger entry exists.")
else:
    print("CHECK: expected exactly one ledger entry.")


In [0]:

from pyspark.sql import functions as F
from delta.tables import DeltaTable

incremental_path = (
    "abfss://bronze@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/incremental/"
)

silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

df = spark.read.option('multiLine', True).json(incremental_path)

# Inspect the delete event and exclude the tombstone
df.select(
    "partition",
    "offset",
    "is_tombstone",
    "event.payload.op",
    "event.payload.before.customer_id",
    "key"
).filter(
    (F.col("offset") == 12) | (F.col("offset") == 13)
).show(truncate=False)


In [0]:

from pyspark.sql import functions as F

incremental_path = (
    "abfss://bronze@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/incremental/"
)

df = spark.read.json(incremental_path)

# Parse the before payload when it is a JSON string
before_type = df.schema["event"].dataType["payload"].dataType["before"].dataType

if isinstance(before_type, __import__("pyspark").sql.types.StringType):
    df = df.withColumn(
        "_before",
        F.from_json(
            F.col("event.payload.before"),
            "customer_id BIGINT, customer_name STRING, city STRING, updated_at BIGINT"
        )
    )
else:
    df = df.withColumn("_before", F.col("event.payload.before"))

df.select(
    "partition",
    "offset",
    "is_tombstone",
    F.col("event.payload.op").alias("operation"),
    F.col("_before.customer_id").alias("deleted_customer_id"),
    "key"
).filter(
    F.col("offset").isin(12, 13)
).orderBy("offset").show(truncate=False)


In [0]:
from pyspark.sql import functions as F

ledger_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/metadata/processed_events/"
)

ledger_df = spark.read.format("delta").load(ledger_path)

ledger_df.filter(
    (F.col("topic") == "retail.public.customers") &
    (F.col("kafka_partition") == 0) &
    (F.col("kafka_offset") == 12)
).show(truncate=False)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

silver_df = spark.read.format("delta").load(silver_path)

print("Before delete:")
silver_df.filter(F.col("customer_id") == 4).show(truncate=False)

DeltaTable.forPath(spark, silver_path).delete(
    "customer_id = 4"
)

print("After delete:")
spark.read.format("delta").load(silver_path) \
    .filter(F.col("customer_id") == 4) \
    .show(truncate=False)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

ledger_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/metadata/processed_events/"
)

ledger_df = spark.read.format("delta").load(ledger_path)

already_processed = ledger_df.filter(
    (F.col("topic") == "retail.public.customers") &
    (F.col("kafka_partition") == 0) &
    (F.col("kafka_offset") == 12)
).limit(1).count()

if already_processed == 0:
    delete_record = spark.createDataFrame(
        [(
            "retail.public.customers",
            0,
            12,
            4,
            "d"
        )],
        """
        topic STRING,
        kafka_partition INT,
        kafka_offset BIGINT,
        customer_id INT,
        operation STRING
        """
    ).withColumn("processed_at", F.current_timestamp())

    delete_record.write.format("delta").mode("append").save(ledger_path)
    print("Delete event recorded in ledger.")
else:
    print("Delete event already exists in ledger; no duplicate added.")

spark.read.format("delta").load(ledger_path).filter(
    (F.col("topic") == "retail.public.customers") &
    (F.col("kafka_partition") == 0) &
    (F.col("kafka_offset") == 12)
).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

silver_path = (
    "abfss://silver@nyctaxiyellow2026pro.dfs.core.windows.net/"
    "cdc/customers/"
)

df_silver = spark.read.format("delta").load(silver_path)

print("Silver schema:")
df_silver.printSchema()

print("Total Silver customers:", df_silver.count())

print("Customer 4 check — expected zero rows:")
df_silver.filter(
    F.col("customer_id") == 4
).show(truncate=False)

print("Current Silver customer records:")
df_silver.orderBy("customer_id").show(truncate=False)